# 再現の確認（embedding_comparison）

`src/embedding_comparison/README.md` の工程 1〜3 を順に流し、`data/derived/embedding_comparison/` の全ファイルがコミットされたものとバイト単位で一致するかを確かめる。

- どこで開いてもよい。リポジトリの一番上を探し、スクリプトはそこから流す。
- `fetch_mat2vec.py`（mat2vec を落とす。約470MB）は流さない。`data/cache/mat2vec/` が無ければ先に流しておく。
- scispaCy の語ベクトル表は `data/cache/scispacy/` を使う（無ければ工程1 が shibayama_novelty の工程2 と同じく落とす）。
- 流す前に作業ツリーの `data/derived/embedding_comparison/` がコミットされた状態であることを前提にする。

In [ ]:
import hashlib
import subprocess
import sys
from pathlib import Path

# notebooks/reproduce/ で開いても動くよう、上にたどってリポジトリの一番上を探す
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "embedding_comparison" / "embed_titles.py").exists())
DERIVED = ROOT / "data" / "derived" / "embedding_comparison"

STEPS = [
    "embed_titles",
    "compute_novelty",
    "compare_embeddings",
]


def snapshot() -> dict[str, str]:
    return {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(DERIVED.rglob("*")) if p.is_file() and p.name != ".DS_Store"}


before = snapshot()
print(f"流す前: {len(before)} ファイル")

In [ ]:
for i, name in enumerate(STEPS, start=1):
    print(f"[{i}] {name}")
    res = subprocess.run([sys.executable, f"src/embedding_comparison/{name}.py"], cwd=ROOT,
                         capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"工程{i} {name} が失敗した")

In [ ]:
after = snapshot()
changed = sorted(k for k in before.keys() & after.keys() if before[k] != after[k])
added = sorted(after.keys() - before.keys())
removed = sorted(before.keys() - after.keys())
for label, items in [("変わった", changed), ("増えた", added), ("消えた", removed)]:
    for k in items:
        print(f"{label}: {k}")
print(f"一致 {len(before.keys() & after.keys()) - len(changed)} / {len(after)} ファイル")
assert not (changed or added or removed), "コミットされた出力と一致しない"
print("✅ すべての出力がコミットされたものと一致した")